# MO14 Stepping Up - Error Identification & Correction

This notebook reads the MO14-Stepping-Up.xlsx financial model, identifies 10 formula errors
in the spreadsheet, builds a corrected Python financial model, and answers 10 multiple-choice
questions based on the corrected outputs.

In [ ]:
import numpy as np
import openpyxl
from datetime import date
import os

# Load the workbook (formulas and cached values)
data_path = '/workspace/data/MO14-Stepping-Up.xlsx'
if not os.path.exists(data_path):
    data_path = os.path.join(os.path.dirname(os.path.abspath('.')),
                             'environment', 'data', 'MO14-Stepping-Up.xlsx')

wb_formulas = openpyxl.load_workbook(data_path, data_only=False)
wb_values = openpyxl.load_workbook(data_path, data_only=True)
ws_inp = wb_values['Inputs']
ws_calc_f = wb_formulas['Calculations']
ws_out_f = wb_formulas['Outputs']

print(f"Sheets: {wb_formulas.sheetnames}")

# Column layout: E(5)=Year0(base), F(6)=Year1(FY2016), ..., T(20)=Year15(FY2030)
COL_E = 5; COL_F = 6; COL_T = 20
N = 16  # years 0..15

def read_row(row):
    """Read a row of Inputs data for columns E through T."""
    return [ws_inp.cell(row=row, column=c).value or 0 for c in range(COL_E, COL_T+1)]

In [ ]:
# ============================================================
# Read all input assumptions from the Inputs sheet
# ============================================================

# Fiscal year-end dates (June 30) and days per year
fy_dates = [date(2015 + i, 6, 30) for i in range(N + 1)]
days_per_year = np.array([0] + [(fy_dates[i] - fy_dates[i-1]).days for i in range(1, N)])

# Widget volumes (8 widgets, rows 37-44)
volumes = np.array([read_row(r) for r in range(37, 45)])

# Widget category assignments (Inputs D48:D55)
categories = [ws_inp.cell(row=r, column=4).value for r in range(48, 56)]
print(f"Widget categories: {categories}")

# CPI rates
cpi = np.array(read_row(28))

# Revenue pricing (Base scenario)
# Code 2000 -> High value, 3000 -> Mid value, 4000 -> Low value
base_price = {2000: np.array(read_row(60)),
              3000: np.array(read_row(61)),
              4000: np.array(read_row(62))}

# Cost of Sales rates
cos_rate = {2000: np.array(read_row(70)),  # 60% for High
            3000: np.array(read_row(71)),   # 65% for Mid
            4000: np.array(read_row(72))}   # 70% for Low

# Overheads
sales_cost_base = ws_inp.cell(row=75, column=COL_E).value   # $14/unit
marketing_pct = np.array(read_row(76))
other_costs_base = ws_inp.cell(row=77, column=COL_E).value  # $750m

# Working capital days
ar_days = np.array(read_row(80))
inv_days = np.array(read_row(81))
ap_days = np.array(read_row(82))

# Capital expenditure
oneoff_capex = np.array(read_row(86))
regular_capex = np.array(read_row(87))
total_capex = oneoff_capex + regular_capex

# Depreciation parameters
acct_life = ws_inp.cell(row=90, column=COL_E).value       # 7 years
tax_depr_rate = ws_inp.cell(row=91, column=COL_E).value   # 20%

# Interest rates
cash_rate = np.array(read_row(95))
tl1_rate = np.array(read_row(96))
tl2_rate = np.array(read_row(97))
tl3_rate = np.array(read_row(98))
revolver_rate_arr = np.array(read_row(99))

# Term loan drawdowns/repayments
tl1_draw = np.array(read_row(102)); tl1_repay = np.array(read_row(103))
tl2_draw = np.array(read_row(104)); tl2_repay = np.array(read_row(105))
tl3_draw = np.array(read_row(106)); tl3_repay = np.array(read_row(107))

# Equity
equity_contrib = np.array(read_row(108))
equity_dist_pct = np.array(read_row(109))

# Tax
tax_rate = ws_inp.cell(row=32, column=COL_E).value  # 30%
print(f"Tax rate: {tax_rate}, Acct life: {acct_life}, Tax depr rate: {tax_depr_rate}")

In [ ]:
# ============================================================
# Read question files
# ============================================================
questions = {}
data_dir = os.path.dirname(data_path)
for i in range(1, 11):
    qfile = os.path.join(data_dir, f'question{i}.txt')
    if os.path.exists(qfile):
        with open(qfile) as f:
            questions[i] = f.read().strip()
        print(f"Q{i}: {questions[i][:80]}...")

In [ ]:
# ============================================================
# ERROR 1 (Q1): Volume summary SUMIF - type mismatch & shifted ranges
# Rows 18-20 in Calculations use SUMIF to aggregate volumes by category.
# BUG: C18 = Inputs!C$60 = 2000 (numeric code), but lookup array D8:D15
#       contains text labels ("Low value", "Mid value", "High value").
#       SUMIF(text_array, number) never matches -> always returns 0.
#       Also rows 19-20 have shifted lookup ranges ($D9:$D16, $D10:$D17).
# FIX: Use text category names as criteria, with consistent D8:D15 range.
# ============================================================

# Show the buggy formulas
for r in [18, 19, 20]:
    criterion = ws_calc_f.cell(row=r, column=3).value  # C18, C19, C20
    formula = ws_calc_f.cell(row=r, column=COL_F).value
    print(f"Row {r}: criterion={criterion!r}, formula={formula!r}")

# Corrected: sum volumes by text category
vol_by_cat = {}
for cat in ['High value', 'Mid value', 'Low value']:
    vol_by_cat[cat] = np.zeros(N)
    for i, c in enumerate(categories):
        if c == cat:
            vol_by_cat[cat] += volumes[i]

# Code mapping: 2000->High, 3000->Mid, 4000->Low
vol_high = vol_by_cat['High value']  # Row 18
vol_mid = vol_by_cat['Mid value']    # Row 19
vol_low = vol_by_cat['Low value']    # Row 20
total_vol = volumes.sum(axis=0)

print(f"\nQ1 Answer: Low value total volume = {vol_low[1:].sum():.2f}")

In [ ]:
# ============================================================
# ERROR 2 (Q2): Scenario selector IF condition has trailing space
# Revenue pricing rows 26-28 use: IF($E$23="Base ", base_price, growth_price)
# BUG: "Base " (trailing space) never matches E23 which is "Base" (no space).
#       So the model always uses Growth pricing instead of Base pricing.
# FIX: Remove trailing space -> IF($E$23="Base", ...)
# ============================================================

# Show the buggy formula
print(f"E23 (scenario): {ws_calc_f.cell(row=23, column=COL_E).value!r}")
print(f"Row 26 formula: {ws_calc_f.cell(row=26, column=COL_F).value!r}")
# Note: 'Base ' with trailing space != 'Base'

# Corrected: use Base pricing
price_high = base_price[2000]
price_mid = base_price[3000]
price_low = base_price[4000]

# Revenue = volume * price for each category
rev_high = vol_high * price_high
rev_mid = vol_mid * price_mid
rev_low = vol_low * price_low
total_revenue = rev_high + rev_mid + rev_low

# Cost of Sales
cos_high_val = rev_high * cos_rate[2000]
cos_mid_val = rev_mid * cos_rate[3000]
cos_low_val = rev_low * cos_rate[4000]
total_cos = cos_high_val + cos_mid_val + cos_low_val
gross_profit = total_revenue - total_cos

print(f"\nQ2 Answer: Total revenue FY2023 (year 8) = {total_revenue[8]:.2f}")

In [ ]:
# ============================================================
# ERROR 3 (Q3): Other costs use fixed CPI instead of year-specific
# Row 63 formula: G63 = F63*(1+Inputs!$F$28)
# BUG: $F$28 is locked to column F (Year 1 CPI = 4%), so all years
#       compound at 4% instead of switching to 3% from year 4 onwards.
# FIX: Use year-specific CPI: G63 = F63*(1+Inputs!G$28)
# ============================================================

print(f"F63 formula: {ws_calc_f.cell(row=63, column=COL_F).value!r}")
print(f"G63 formula: {ws_calc_f.cell(row=63, column=COL_F+1).value!r}")
print(f"Note: Inputs!$F$28 is always year 1 CPI = {cpi[1]}")
print(f"But year 4+ CPI should be {cpi[4]}")

# Sales overhead per unit (grows with CPI)
overhead_per_unit = np.zeros(N)
overhead_per_unit[0] = sales_cost_base
for t in range(1, N):
    overhead_per_unit[t] = overhead_per_unit[t-1] * (1 + cpi[t])
sales_expense = overhead_per_unit * total_vol

# Marketing expense
marketing_expense = marketing_pct * total_revenue

# Other costs (FIXED: year-specific CPI)
other_costs = np.zeros(N)
other_costs[0] = other_costs_base
for t in range(1, N):
    other_costs[t] = other_costs[t-1] * (1 + cpi[t])  # FIX: was (1 + cpi[1])

total_expenses = sales_expense + marketing_expense + other_costs

print(f"\nQ3 Answer: Total expenses FY2025 (year 10) = {total_expenses[10]:.2f}")

In [ ]:
# ============================================================
# ERROR 4 (Q4): NWC formula references wrong row
# Row 87: F87 = F78 - F77 - F84
# BUG: F77 = Inputs!F$81 = inventory days (42), which is a days count,
#       not a dollar amount. Should reference F74 (accounts receivable).
# FIX: F87 = F74 + F78 - F84 (AR + Inventory - AP)
# ============================================================

print(f"NWC formula (buggy): {ws_calc_f.cell(row=87, column=COL_F).value!r}")
print("F78=Inventory, F77=Inventory days (42!), F84=AP")
print("Should be: F74 + F78 - F84 (AR + Inventory - AP)")

# Corrected working capital
ar = np.zeros(N)
inventory = np.zeros(N)
ap = np.zeros(N)
for t in range(1, N):
    d = days_per_year[t]
    ar[t] = total_revenue[t] * (ar_days[t] / d)
    inventory[t] = total_cos[t] * (inv_days[t] / d)
    ap[t] = (total_cos[t] + total_expenses[t]) * (ap_days[t] / d)

nwc = ar + inventory - ap
change_nwc = np.zeros(N)
for t in range(1, N):
    change_nwc[t] = nwc[t-1] - nwc[t]

print(f"\nQ4 Answer: Change in NWC FY2019 (year 4) = {change_nwc[4]:.2f}")

In [ ]:
# ============================================================
# ERROR 5 (Q5): Depreciation tranches depreciate forever
# Row 103+: (F$2 >= $C103) * (INDEX($F$97:$T$97, $C103) / $E$99)
# BUG: Condition (year >= purchase_year) is always true once started,
#       so each tranche depreciates beyond its useful life of 7 years.
# FIX: Add upper bound: (F$2 >= $C103) AND (F$2 < $C103 + acct_life)
# ============================================================

print(f"Tranche formula: {ws_calc_f.cell(row=103, column=COL_T).value!r}")
print(f"Accounting life: {acct_life} years")
print("BUG: No upper bound on depreciation period")

# Corrected depreciation with useful life cap
acct_dep_tranches = np.zeros((N, N))
for t in range(1, N):
    if total_capex[t] > 0:
        annual_dep = total_capex[t] / acct_life
        for y in range(t, min(t + int(acct_life), N)):  # Cap at useful life
            acct_dep_tranches[t, y] = annual_dep

acct_dep = acct_dep_tranches.sum(axis=0)
acct_nbv = np.zeros(N)
for t in range(1, N):
    opening = acct_nbv[t-1]
    acct_dep[t] = min(acct_dep[t], opening + total_capex[t])
    acct_nbv[t] = opening + total_capex[t] - acct_dep[t]

print(f"\nQ5 Answer: Closing accounting NBV FY2030 = {acct_nbv[15]:.2f}")

In [ ]:
# ============================================================
# ERROR 6 (Q6): Tax depreciation opening references accounting NBV
# Row 124: F124 = F93 (opening accounting NBV)
# BUG: Should reference E127 (prior year closing tax NBV), not F93.
# FIX: F124 = E127 (carry forward closing tax NBV from prior year)
# ============================================================

print(f"Tax dep opening formula: {ws_calc_f.cell(row=124, column=COL_F).value!r}")
print("BUG: =F93 references accounting NBV opening")
print("FIX: Should reference E127 (prior closing tax NBV)")

# Corrected tax depreciation (diminishing value)
tax_nbv = np.zeros(N)
tax_dep = np.zeros(N)
for t in range(1, N):
    opening_tax = tax_nbv[t-1]  # FIX: previous year's closing tax NBV
    tax_dep[t] = tax_depr_rate * (opening_tax + total_capex[t])
    tax_nbv[t] = opening_tax + total_capex[t] - tax_dep[t]

print(f"\nQ6 Answer: Total tax depreciation = {tax_dep[1:].sum():.2f}")

In [ ]:
# ============================================================
# EBITDA and Term Loan Balances (needed for Q7-Q10)
# ============================================================

total_overhead = sales_expense + marketing_expense + other_costs
ebitda = gross_profit - total_overhead

# Term loan balances
tl1_bal = np.zeros(N)
tl2_bal = np.zeros(N)
tl3_bal = np.zeros(N)
for t in range(1, N):
    tl1_bal[t] = tl1_bal[t-1] + tl1_draw[t] + tl1_repay[t]
    tl2_bal[t] = tl2_bal[t-1] + tl2_draw[t] + tl2_repay[t]
    tl3_bal[t] = tl3_bal[t-1] + tl3_draw[t] + tl3_repay[t]

total_tl = tl1_bal + tl2_bal + tl3_bal
tl_cf = tl1_draw + tl1_repay + tl2_draw + tl2_repay + tl3_draw + tl3_repay

# ERROR 8 (Q8): TL3 interest references TL2 data in cols P-T
# Row 202 (TL3 interest): cols P-T have =-SUM(P192:P193)*P201
# BUG: P192:P193 are TL2 opening/drawdown. Should be P199:P200 (TL3).
# FIX: Use correct TL3 balances for interest calculation.
print(f"TL3 interest col F: {ws_calc_f.cell(row=202, column=COL_F).value!r}")
print(f"TL3 interest col P: {ws_calc_f.cell(row=202, column=16).value!r}")
print("BUG: Col P+ references rows 192-193 (TL2) instead of 199-200 (TL3)")

# Corrected TL interest
tl1_interest = np.zeros(N)
tl2_interest = np.zeros(N)
tl3_interest = np.zeros(N)
for t in range(1, N):
    tl1_interest[t] = -(tl1_bal[t-1] + tl1_draw[t]) * tl1_rate[t]
    tl2_interest[t] = -(tl2_bal[t-1] + tl2_draw[t]) * tl2_rate[t]
    tl3_interest[t] = -(tl3_bal[t-1] + tl3_draw[t]) * tl3_rate[t]  # FIX: correct TL3 data

total_tl_interest = tl1_interest + tl2_interest + tl3_interest
print(f"\nQ8 Answer: Total TL interest FY2028 = {abs(total_tl_interest[13]):.2f}")

In [ ]:
# ============================================================
# Iterative model: cash, revolver, distributions
# (circular reference: interest depends on cash, cash depends on interest)
# ============================================================

cash = np.zeros(N)
revolver_bal = np.zeros(N)
distributions = np.zeros(N)

for iteration in range(200):
    old_cash = cash.copy()
    
    # Interest revenue on cash at bank
    interest_revenue = np.zeros(N)
    revolver_interest = np.zeros(N)
    for t in range(1, N):
        interest_revenue[t] = cash[t-1] * cash_rate[t]
        revolver_interest[t] = -revolver_bal[t-1] * revolver_rate_arr[t]
    
    total_interest = interest_revenue + total_tl_interest + revolver_interest
    
    # P&L
    pbt = ebitda - acct_dep + total_interest
    income_tax = -tax_rate * pbt
    npat = pbt + income_tax
    
    # ERROR 9 (Q9): Tax timing difference has wrong sign
    # Row 224 = -acct_dep, Row 225 = -tax_dep
    # BUG: F226 = F224 - F225 = (-acct_dep) - (-tax_dep) = tax_dep - acct_dep
    # FIX: F226 = F225 - F224 = acct_dep - tax_dep
    # When tax_dep > acct_dep, the adjustment should REDUCE taxable income.
    timing_diff = acct_dep - tax_dep  # FIX: reversed sign
    adjusted_pbt = pbt + timing_diff
    tax_payable = -adjusted_pbt * tax_rate
    
    # ERROR 7 (Q7): Cash flow signs
    # BUG in Outputs row 76: =- Calculations!F210 (double negation of interest)
    # FIX: =Calculations!F210 (interest is already negative for expense)
    # BUG in Outputs row 80: =Calculations!F97 (capex as positive in investing)
    # FIX: =-Calculations!F97 (capex should be negative cash outflow)
    operating_cf = ebitda + change_nwc + tax_payable + total_interest  # FIX: correct interest sign
    investing_cf = -total_capex  # FIX: negate capex
    free_cf = operating_cf + investing_cf
    
    # Cash flow waterfall
    for t in range(1, N):
        opening_cash = cash[t-1]
        fcf_avail = opening_cash + free_cf[t] + equity_contrib[t] + tl_cf[t]
        
        if fcf_avail < 0:
            rev_draw = -fcf_avail
            rev_pay = 0
        else:
            rev_pay = -min(revolver_bal[t-1], fcf_avail) if revolver_bal[t-1] > 0 else 0
            rev_draw = 0
        
        revolver_bal[t] = revolver_bal[t-1] + rev_draw + rev_pay
        cash_after = fcf_avail + rev_draw + rev_pay
        distributions[t] = max(0, cash_after * equity_dist_pct[t])
        cash[t] = cash_after - distributions[t]
    
    if np.max(np.abs(cash - old_cash)) < 0.0001:
        print(f"Model converged after {iteration+1} iterations")
        break

In [ ]:
# ============================================================
# Q7: Operating + Investing CF for FY2016
# Note: Q7 answer is based on fixes Q1-Q7 only, BEFORE the Q9 fix.
# The Q9 timing difference correction changes tax_payable.
# ============================================================

print(f"Interest formula (buggy): {ws_out_f.cell(row=76, column=COL_F).value!r}")
print("BUG: =-Calculations!F210 double-negates interest")
print(f"Capex formula (buggy): {ws_out_f.cell(row=80, column=COL_F).value!r}")
print("BUG: =Calculations!F97 shows capex as positive (should be negative outflow)")

# Compute Q7 with buggy Q9 (timing_diff = tax_dep - acct_dep, wrong sign)
timing_diff_buggy = tax_dep - acct_dep  # Buggy Q9 version
adjusted_pbt_buggy = pbt + timing_diff_buggy
tax_payable_buggy = -adjusted_pbt_buggy * tax_rate
operating_cf_q7 = ebitda + change_nwc + tax_payable_buggy + total_interest
q7_value = operating_cf_q7[1] + investing_cf[1]

print(f"\nQ7 Answer: Op+Inv CF FY2016 = {q7_value:.2f}")

In [ ]:
# ============================================================
# Q9: Tax payable FY2019
# ============================================================

print(f"Timing diff formula (buggy): {ws_calc_f.cell(row=226, column=COL_F).value!r}")
print(f"Row 224 = -acct_dep: {ws_calc_f.cell(row=224, column=COL_F).value!r}")
print(f"Row 225 = -tax_dep: {ws_calc_f.cell(row=225, column=COL_F).value!r}")
print("BUG: F226 = F224-F225 = tax_dep - acct_dep (wrong direction)")
print("FIX: F226 = F225-F224 = acct_dep - tax_dep")

print(f"\nQ9 Answer: Tax payable FY2019 = {abs(tax_payable[4]):.2f}")

In [ ]:
# ============================================================
# Q10: Balance sheet - Total net assets FY2030
# ERROR 10: Balance sheet doesn't balance because of all prior errors.
# After fixing Q1-Q9, the balance sheet should balance.
# ============================================================

# Retained earnings
retained = np.zeros(N)
equity_bal = np.zeros(N)
for t in range(1, N):
    retained[t] = retained[t-1] + npat[t] - distributions[t]
    equity_bal[t] = equity_bal[t-1] + equity_contrib[t]

# Deferred tax asset/liability
dep_timing_cum = np.zeros(N)
for t in range(1, N):
    dep_timing_cum[t] = dep_timing_cum[t-1] + timing_diff[t]

dt_balance = -tax_rate * dep_timing_cum
dta = np.where(dt_balance < 0, -dt_balance, 0)  # Deferred tax asset
dtl = np.where(dt_balance > 0, dt_balance, 0)    # Deferred tax liability

# Net assets = Total assets - Total liabilities
total_assets = cash + ar + inventory + acct_nbv + dta
total_liabilities = ap + revolver_bal + total_tl + dtl
net_assets = total_assets - total_liabilities
total_equity = equity_bal + retained

# Verify balance sheet balances
for t in [1, 4, 8, 15]:
    diff = abs(net_assets[t] - total_equity[t])
    print(f"Year {t}: Net Assets={net_assets[t]:.2f}, Total Equity={total_equity[t]:.2f}, Diff={diff:.4f}")

print(f"\nQ10 Answer: Total net assets FY2030 = {net_assets[15]:.2f}")

In [ ]:
# ============================================================
# Match computed values to multiple-choice options
# ============================================================

option_map = {
    1: {'a': 4829.07, 'b': 4829.28, 'c': 4829.39, 'd': 4829.42, 'e': 4829.68, 'f': 4829.72},
    2: {'a': 68560.10, 'b': 68560.28, 'c': 68560.47, 'd': 68560.59, 'e': 68560.71, 'f': 68560.98},
    3: {'a': 18517.04, 'b': 18517.12, 'c': 18517.37, 'd': 18517.46, 'e': 18517.68, 'f': 18517.97},
    4: {'a': -168.22, 'b': -168.36, 'c': -168.41, 'd': -168.68, 'e': -168.79, 'f': -168.81},
    5: {'a': 6211.04, 'b': 6211.14, 'c': 6211.27, 'd': 6211.39, 'e': 6211.57, 'f': 6211.62},
    6: {'a': 34443.04, 'b': 34443.14, 'c': 34443.38, 'd': 34443.57, 'e': 34443.77, 'f': 34443.95},
    7: {'a': -9428.17, 'b': -9428.36, 'c': -9428.48, 'd': -9428.57, 'e': -9428.66, 'f': -9428.79},
    8: {'a': 996, 'b': 997, 'c': 998, 'd': 999, 'e': 1000, 'f': 1001},
    9: {'a': 1111.24, 'b': 1111.47, 'c': 1111.56, 'd': 1111.65, 'e': 1111.80, 'f': 1111.92},
    10: {'a': 8085.44, 'b': 8085.55, 'c': 8085.66, 'd': 8085.77, 'e': 8085.88, 'f': 8085.99},
}

computed = {
    1: vol_low[1:].sum(),
    2: total_revenue[8],
    3: total_expenses[10],
    4: change_nwc[4],
    5: acct_nbv[15],
    6: tax_dep[1:].sum(),
    7: q7_value,
    8: abs(total_tl_interest[13]),
    9: abs(tax_payable[4]),
    10: net_assets[15],
}

answers = {}
for q in range(1, 11):
    val = computed[q]
    best_letter = min(option_map[q], key=lambda k: abs(option_map[q][k] - val))
    best_val = option_map[q][best_letter]
    diff = abs(val - best_val)
    answers[f'question{q}'] = best_letter.upper()
    print(f"Q{q}: computed={val:.2f}, best match={best_letter.upper()} ({best_val}), diff={diff:.4f}")

print(f"\nanswers = {answers}")

In [ ]:
# Final answers dictionary
answers = {
    'question1': answers['question1'],
    'question2': answers['question2'],
    'question3': answers['question3'],
    'question4': answers['question4'],
    'question5': answers['question5'],
    'question6': answers['question6'],
    'question7': answers['question7'],
    'question8': answers['question8'],
    'question9': answers['question9'],
    'question10': answers['question10'],
}

print("Final answers:")
for k, v in sorted(answers.items()):
    print(f"  {k}: {v}")